# 🛡️ Day 03a: Security Best Practices — OWASP Top 10 Defense

---

## 🎯 What You'll Master Today
- OWASP Top 10 web vulnerabilities
- SQL injection prevention
- XSS and CSRF protection
- Rate limiting implementation
- Input sanitization and validation
- Secure FastAPI configuration

---

## 🎭 The Castle Defense Analogy

Your API is a castle. Security isn't one giant wall — it's **layers**: moat (CORS), drawbridge (auth), guards (validation), watchtowers (logging), and a dungeon (rate limiting for attackers).

```
╔═══════════════════════════════════════════════════════════════════╗
║  OWASP TOP 10 (2021) — MOST CRITICAL WEB RISKS                 ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  A01: Broken Access Control      → check permissions properly    ║
║  A02: Cryptographic Failures     → hash passwords, use HTTPS    ║
║  A03: Injection (SQL/NoSQL/CMD)  → parameterized queries, ORM   ║
║  A04: Insecure Design            → threat modeling, principles  ║
║  A05: Security Misconfiguration  → no defaults, no debug mode   ║
║  A06: Vulnerable Components      → update dependencies          ║
║  A07: Auth Failures              → strong passwords, MFA        ║
║  A08: Data Integrity Failures    → verify CI/CD, sign JWTs     ║
║  A09: Logging Failures           → monitor, alert, audit trail  ║
║  A10: SSRF                       → validate/restrict outgoing   ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# SQL Injection Prevention
# ============================================================

print("""
# ❌ VULNERABLE — Never do this!
query = f"SELECT * FROM users WHERE username = '{user_input}'"
# If user_input = "' OR '1'='1" → returns ALL users!
# If user_input = "'; DROP TABLE users; --" → deletes table!

# ✅ SAFE — Parameterized queries
query = "SELECT * FROM users WHERE username = :username"
result = db.execute(query, {"username": user_input})

# ✅ SAFE — SQLAlchemy ORM (automatically parameterizes)
user = db.query(User).filter(User.username == user_input).first()

# ✅ SAFE — Pydantic validation (strips dangerous input)
class UserQuery(BaseModel):
    username: str = Field(..., min_length=1, max_length=50, pattern=r'^[a-zA-Z0-9_]+$')
""")

print("Defense layers against injection:")
print("  1. Pydantic validation ← first line of defense")
print("  2. SQLAlchemy ORM ← auto-parameterizes queries")
print("  3. Principle of least privilege ← DB user with minimal perms")

In [ ]:
# ============================================================
# XSS (Cross-Site Scripting) Prevention
# ============================================================
import html

# XSS: attacker injects JavaScript that runs in other users' browsers
malicious_input = '<script>alert("hacked")</script>'

# ❌ VULNERABLE — Rendering raw user input
# return HTMLResponse(f"<h1>Welcome, {user_input}</h1>")

# ✅ SAFE — Escape HTML
safe_output = html.escape(malicious_input)
print(f"Raw input:     {malicious_input}")
print(f"Escaped:       {safe_output}")

print("")
# ✅ For APIs returning JSON (FastAPI default): XSS is not a direct risk
# because JSON responses aren't rendered as HTML by the browser.
# BUT if your frontend renders user-generated content:

print("XSS prevention checklist:")
print("  1. FastAPI returns JSON → not directly vulnerable")
print("  2. Frontend must escape user content before rendering")
print("  3. Set Content-Security-Policy headers")
print("  4. Use httpOnly cookies (JS can't read them)")
print("  5. Sanitize HTML input with bleach library")

In [ ]:
# ============================================================
# Rate Limiting — Prevent Abuse
# ============================================================
from fastapi import FastAPI, Request, HTTPException
from fastapi.testclient import TestClient
from collections import defaultdict
import time

app = FastAPI()

# Simple in-memory rate limiter (use Redis in production)
class RateLimiter:
    def __init__(self, max_requests: int, window_seconds: int):
        self.max_requests = max_requests
        self.window = window_seconds
        self.requests = defaultdict(list)
    
    def is_allowed(self, client_id: str) -> bool:
        now = time.time()
        # Remove expired timestamps
        self.requests[client_id] = [
            t for t in self.requests[client_id] if now - t < self.window
        ]
        if len(self.requests[client_id]) >= self.max_requests:
            return False
        self.requests[client_id].append(now)
        return True

limiter = RateLimiter(max_requests=5, window_seconds=60)  # 5 req/min

@app.middleware("http")
async def rate_limit_middleware(request: Request, call_next):
    client_ip = request.client.host
    if not limiter.is_allowed(client_ip):
        return JSONResponse(
            status_code=429,
            content={"detail": "Too many requests. Please try again later."},
            headers={"Retry-After": "60"}
        )
    response = await call_next(request)
    return response

from fastapi.responses import JSONResponse

@app.get("/api/data")
def get_data():
    return {"data": "here"}

client = TestClient(app)
for i in range(7):
    r = client.get("/api/data")
    status = "✅" if r.status_code == 200 else "❌ 429"
    print(f"Request {i+1}: {status}")

print("\n💡 Production: Use slowapi or redis-based rate limiting")

In [ ]:
# ============================================================
# Security Headers — Defense in Depth
# ============================================================

app2 = FastAPI()

@app2.middleware("http")
async def add_security_headers(request: Request, call_next):
    response = await call_next(request)
    
    # Prevent clickjacking
    response.headers["X-Frame-Options"] = "DENY"
    
    # Prevent MIME-type sniffing
    response.headers["X-Content-Type-Options"] = "nosniff"
    
    # XSS protection (legacy browsers)
    response.headers["X-XSS-Protection"] = "1; mode=block"
    
    # Content Security Policy
    response.headers["Content-Security-Policy"] = "default-src 'self'"
    
    # Force HTTPS
    response.headers["Strict-Transport-Security"] = "max-age=31536000; includeSubDomains"
    
    # Referrer policy
    response.headers["Referrer-Policy"] = "strict-origin-when-cross-origin"
    
    return response

@app2.get("/secure")
def secure_endpoint():
    return {"secure": True}

client2 = TestClient(app2)
r = client2.get("/secure")
print("Security headers:")
for header in ["X-Frame-Options", "X-Content-Type-Options", "Content-Security-Policy"]:
    print(f"  {header}: {r.headers.get(header)}")

In [ ]:
# ============================================================
# Input Validation — Pydantic as Security Layer
# ============================================================
from pydantic import BaseModel, Field, field_validator
import re

class SecureUserInput(BaseModel):
    # Length limits prevent buffer overflow / DoS
    username: str = Field(..., min_length=3, max_length=30, pattern=r'^[a-zA-Z0-9_]+$')
    email: str = Field(..., max_length=254)
    bio: str = Field(default="", max_length=500)
    age: int = Field(..., ge=13, le=150)
    
    @field_validator('email')
    @classmethod
    def validate_email(cls, v):
        pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
        if not re.match(pattern, v):
            raise ValueError('Invalid email format')
        return v.lower()
    
    @field_validator('bio')
    @classmethod
    def sanitize_bio(cls, v):
        # Strip HTML tags
        clean = re.sub(r'<[^>]+>', '', v)
        return clean.strip()

# Test
from pydantic import ValidationError

tests = [
    {"username": "sarthak", "email": "s@m.com", "age": 25},
    {"username": "a' OR 1=1--", "email": "s@m.com", "age": 25},  # SQL injection attempt
    {"username": "<script>alert(1)</script>", "email": "s@m.com", "age": 25},  # XSS attempt
]

for t in tests:
    try:
        user = SecureUserInput(**t)
        print(f"✅ Valid: {user.username}")
    except ValidationError as e:
        print(f"❌ Blocked: {t['username'][:30]} — {e.errors()[0]['msg']}")

In [ ]:
# ============================================================
# Secure FastAPI Configuration Checklist
# ============================================================

print("""
Production Security Checklist:

1. ENVIRONMENT VARIABLES for secrets:
   SECRET_KEY = os.getenv("SECRET_KEY")
   DATABASE_URL = os.getenv("DATABASE_URL")
   # Never commit secrets to Git!

2. DISABLE DOCS in production:
   app = FastAPI(
       docs_url=None if PRODUCTION else "/docs",
       redoc_url=None if PRODUCTION else "/redoc",
   )

3. CORS — specific origins only:
   allow_origins=["https://myapp.com"]  # Not "*"!

4. HTTPS only:
   - TLS termination at load balancer
   - HSTS header

5. RATE LIMITING:
   - Per-IP or per-user limits
   - Stricter on auth endpoints

6. INPUT VALIDATION:
   - Pydantic models for all request bodies
   - Max length on all string fields
   - Regex patterns for structured data

7. ERROR MESSAGES — don't leak internals:
   ❌ "PostgreSQL error: relation 'users' column 'ssn' does not exist"
   ✅ "An error occurred. Please try again."

8. DEPENDENCY AUDIT:
   pip audit              # Check for known vulnerabilities
   pip install safety && safety check

9. LOGGING — log security events:
   - Failed login attempts
   - Rate limit hits
   - Permission denied
   - But NEVER log passwords or tokens!
""")

---

## 🗺️ Security Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  SECURITY BEST PRACTICES                                         ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  INJECTION:  Pydantic validation + ORM + parameterized queries  ║
║  XSS:        JSON APIs safe by default + CSP headers            ║
║  CSRF:       SameSite cookies + anti-CSRF tokens                ║
║  AUTH:       bcrypt + JWT + refresh tokens + scopes             ║
║                                                                   ║
║  RATE LIMIT: Redis-backed limiter, 429 Too Many Requests       ║
║  HEADERS:    X-Frame-Options, CSP, HSTS, nosniff               ║
║  SECRETS:    Environment variables, never in code               ║
║  ERRORS:     Generic messages to client, details in logs        ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What is SQL injection? | Injecting SQL code through user input. Fix: ORM, parameterized queries |
| 2 | XSS vs CSRF? | XSS: inject scripts in page. CSRF: trick user into performing actions |
| 3 | How to prevent CSRF? | SameSite cookies, CSRF tokens, verify Origin/Referer headers |
| 4 | What is rate limiting? | Limiting requests per client per time window. Prevents DoS |
| 5 | OWASP Top 10? | Top 10 web security risks: injection, broken auth, XSS, CSRF, etc. |
| 6 | Where to store secrets? | Environment variables, secret managers (AWS Secrets, HashiCorp Vault) |
| 7 | What is HSTS? | HTTP Strict Transport Security — forces browsers to use HTTPS |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Defense in depth — multiple security layers          │
## │  • Pydantic validates input (first line of defense)     │
## │  • ORM prevents SQL injection automatically             │
## │  • Rate limiting stops brute force and DoS              │
## │  • Security headers add browser-level protection        │
## │  • Never expose secrets, internals, or stack traces     │
## │  • Audit dependencies for known vulnerabilities         │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Chapter 05 — Async Processing** — async/await, Celery, WebSockets